## เพิ่ม Feature สำหรับการสร้าง model

In [13]:
import json
import cv2
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "data" / "jpg"
MASKPATH = PROJECT_ROOT / "data" / "fruit_masks"
MANIFEST_PATH = PROJECT_ROOT / "data" / "defect_manifest.json"

In [14]:
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    fruits = json.load(f)
    f.close()

fruits

[{'image_id': 'NDM_M1 (1)',
  'grade': 'NDM_M1',
  'image': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\jpg\\NDM_M1\\NDM_M1 (1).jpg',
  'fruit_mask': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\fruit_masks\\NDM_M1\\NDM_M1 (1)_mask.png',
  'defect_mask': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\defect_masks\\NDM_M1\\NDM_M1 (1)_defect.png',
  'yellow_mask': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\defect_masks\\NDM_M1\\NDM_M1 (1)_yellow.png',
  'ink_mask': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\defect_masks\\NDM_M1\\NDM_M1 (1)_ink.png',
  'brightness_threshold': 22,
  'bboxes_xywh': [[552, 1077, 16, 11],
   [1672, 909, 34, 12],
   [1551, 894, 38, 50],
   [1430, 892, 28, 56]]},
 {'image_id': 'NDM_M1 (10)',
  'grade': 'NDM_M1',
  'image': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\jpg\\NDM_M1\\NDM_M1 (10).jpg',
  'fruit_mask': 'c:\\Users\\noppa\\OneDrive\\Desktop\\mango\\data\\fruit_masks\\NDM_M1\\NDM_M1 (10)_mask.png',
  'defect_mask':

In [25]:
def feature_engineering(fruits):
    for fruit in fruits:
        fruit_mask = cv2.imread(
            fruit["fruit_mask"],
            cv2.IMREAD_GRAYSCALE,
        )
        defect_mask = cv2.imread(
            fruit["defect_mask"],
            cv2.IMREAD_GRAYSCALE,
        )

        if fruit_mask is None or defect_mask is None:
            raise FileNotFoundError(
                f"ไม่พบ mask ของ {fruit.get('image_id', 'unknown')}"
            )

        fruit_binary = (fruit_mask > 0).astype(np.uint8)
        defect_binary = (defect_mask > 0).astype(np.uint8)
        fruit_area = np.count_nonzero(fruit_binary)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            defect_binary,
            connectivity=8,
        )
        defect_areas = stats[1:, cv2.CC_STAT_AREA]
        defect_areas = defect_areas[defect_areas > 0]

        contours, _ = cv2.findContours(
            fruit_binary,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        fruit_contour = max(contours, key=cv2.contourArea) if contours else None
        fruit_contour_area = (
            cv2.contourArea(fruit_contour)
            if fruit_contour is not None
            else 0.0
        )
        perimeter = (
            cv2.arcLength(fruit_contour, True)
            if fruit_contour is not None
            else 0.0
        )
        hull_area = (
            cv2.contourArea(cv2.convexHull(fruit_contour))
            if fruit_contour is not None
            else 0.0
        )

        if fruit_contour is not None:
            _, _, width, height = cv2.boundingRect(fruit_contour)
        else:
            width, height = 0, 0

        fruit["defect_percentage"] = (
            float(np.count_nonzero(defect_binary) / fruit_area * 100)
            if fruit_area > 0
            else 0.0
        )
        fruit["number_of_defects"] = int(len(defect_areas))
        fruit["largest_defect_ratio"] = (
            float(defect_areas.max() / fruit_area)
            if fruit_area > 0 and len(defect_areas) > 0
            else 0.0
        )
        fruit["mean_defect_ratio"] = (
            float(defect_areas.mean() / fruit_area)
            if fruit_area > 0 and len(defect_areas) > 0
            else 0.0
        )
        fruit["aspect_ratio"] = (
            float(width / height)
            if height > 0
            else 0.0
        )
        fruit["circularity"] = (
            float(4 * np.pi * fruit_contour_area / perimeter**2)
            if perimeter > 0
            else 0.0
        )
        fruit["solidity"] = (
            float(fruit_contour_area / hull_area)
            if hull_area > 0
            else 0.0
        )

    features = pd.DataFrame(fruits)
    return fruits, features


fruits, features = feature_engineering(fruits)
features.head()

,image_id,grade,image,fruit_mask,defect_mask,yellow_mask,ink_mask,brightness_threshold,bboxes_xywh,defect_percentage,number_of_defects,largest_defect_ratio,mean_defect_ratio,aspect_ratio,circularity,solidity
0,NDM_M1 (1),NDM_M1,c:\Users\noppa\OneDrive\Desktop\mango\data\jpg...,c:\Users\noppa\OneDrive\Desktop\mango\data\fru...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,22,"[[552, 1077, 16, 11], [1672, 909, 34, 12], [15...",0.114730,4,0.000528,0.000287,2.183805,0.602172,0.959802
1,NDM_M1 (10),NDM_M1,c:\Users\noppa\OneDrive\Desktop\mango\data\jpg...,c:\Users\noppa\OneDrive\Desktop\mango\data\fru...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,17,"[[587, 631, 19, 11], [773, 380, 24, 35], [759,...",0.507087,21,0.000777,0.000241,0.501342,0.627031,0.960740
2,NDM_M1 (100),NDM_M1,c:\Users\noppa\OneDrive\Desktop\mango\data\jpg...,c:\Users\noppa\OneDrive\Desktop\mango\data\fru...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,18,"[[1018, 338, 21, 12], [953, 337, 27, 19], [945...",0.151443,7,0.000439,0.000216,0.459836,0.572626,0.951242
3,NDM_M1 (101),NDM_M1,c:\Users\noppa\OneDrive\Desktop\mango\data\jpg...,c:\Users\noppa\OneDrive\Desktop\mango\data\fru...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,20,"[[1734, 961, 30, 75], [1844, 943, 21, 17], [18...",0.165285,7,0.000404,0.000236,2.193258,0.600401,0.957939
4,NDM_M1 (102),NDM_M1,c:\Users\noppa\OneDrive\Desktop\mango\data\jpg...,c:\Users\noppa\OneDrive\Desktop\mango\data\fru...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,c:\Users\noppa\OneDrive\Desktop\mango\data\def...,20,"[[962, 265, 73, 30], [908, 244, 58, 17], [948,...",0.136193,6,0.000377,0.000227,0.455943,0.599318,0.958093
